# RelCheck — Visual Genome Recall Test

Tests whether RelCheck correctly **detects** naturally-occurring relational hallucinations
in BLIP-2 captions, using Visual Genome ground-truth relation annotations.

## How it works
1. Download VG relationships.json (~50 MB compressed)
2. Pick N_IMAGES images with >= MIN_VG_RELS diverse VG relations
3. Generate BLIP-2 captions for those images
4. Extract Llama triples from each caption
5. Match caption triples to VG ground truth — identify **contradictions** (ground truth hallucinations)
6. Run RelCheck VQA detection on each contradiction triple
7. Compute: **recall = hallucinations detected / total ground truth hallucinations**

## What this proves
Synthetic injection tests confirm recall on known fabrications. This confirms recall on **naturally-occurring** captioner hallucinations — a harder, more honest test.


In [ ]:
# ============================================================
# Cell 1 — Setup
# ============================================================
!pip install together Pillow requests -q

import os, json, re, time, random, requests, base64, zipfile, io
from pathlib import Path
from collections import Counter
from io import BytesIO
from PIL import Image
from together import Together
from google.colab import drive

# ── Config ──────────────────────────────────────────────────
TOGETHER_API_KEY = ''   # <-- paste your Together.ai key
N_IMAGES         = 20   # VG images to test
MIN_VG_RELS      = 3    # min VG relations per image (richer = better signal)
CAPTIONER        = 'qwen'   # 'qwen' | 'llava' — captioner to evaluate
SAVE_DIR         = '/content/drive/MyDrive/RelCheck_Data/vg_recall'
VG_DATA_PATH     = f'{SAVE_DIR}/relationships_sample.json'
LOCAL_IMAGES_ZIP = f'{SAVE_DIR}/images2.zip'   # your downloaded VG images zip

LLM_MODEL    = 'meta-llama/Llama-3.3-70B-Instruct-Turbo'
VLM_MODEL    = 'Qwen/Qwen3-VL-8B-Instruct'          # used for VQA verification
# Captioner models (Together.ai VLM API — no local GPU load needed)
_CAPTIONER_MODELS = {
    'qwen':  'Qwen/Qwen3-VL-8B-Instruct',
    'llava': 'llava-hf/llava-1.5-7b-hf',
}
CAPTION_MODEL = _CAPTIONER_MODELS[CAPTIONER]

drive.mount('/content/drive')
os.makedirs(SAVE_DIR, exist_ok=True)
os.environ['TOGETHER_API_KEY'] = TOGETHER_API_KEY
client = Together(api_key=TOGETHER_API_KEY)

def llm_call(messages, model=LLM_MODEL, max_tokens=600, retries=3):
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model, messages=messages, temperature=0.0, max_tokens=max_tokens,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                print(f'  API failed: {e}')
                return None

def vlm_call(messages, max_tokens=10, retries=3):
    return llm_call(messages, model=VLM_MODEL, max_tokens=max_tokens, retries=retries)

print('Setup done.')

# ── Config: use full RelCheck verifier (requires GPU + GDino) ─────────
USE_GDINO = True   # False → plain 3-question VQA (no GPU needed)

GDINO_ID            = 'IDEA-Research/grounding-dino-tiny'
MAVERICK_VLM        = 'Qwen/Qwen3-VL-8B-Instruct'   # same as VLM_MODEL here
DETECTION_THRESHOLD = 0.25

# ── Load GDino (only if USE_GDINO) ────────────────────────────────────
gdino_processor = gdino_model = None
if USE_GDINO:
    try:
        import torch
        from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
        DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f'Loading GroundingDINO on {DEVICE}...')
        gdino_processor = AutoProcessor.from_pretrained(GDINO_ID)
        gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_ID).to(DEVICE)
        print('  GroundingDINO ready.')
    except Exception as e:
        print(f'  GDino load failed: {e} — falling back to plain VQA')
        USE_GDINO = False

# ── RelCheck verifier helpers ─────────────────────────────────────────
import re as _re, base64 as _b64
from io import BytesIO as _BytesIO

_SPATIAL_RELS = {
    'left of','to the left of','to the left',
    'right of','to the right of','to the right',
    'above','on top of','over','on',
    'below','under','beneath','underneath',
    'in front of','behind','in back of',
    'inside','outside',
}

def _classify_rel(rel):
    r = rel.lower().strip()
    if r in _SPATIAL_RELS:
        return 'SPATIAL'
    _ACTION_WORDS = {'riding','holding','carrying','eating','drinking','wearing',
                     'pushing','pulling','walking','running','sitting','standing',
                     'playing','using','throwing','catching','driving','feeding',
                     'reading','watching','kicking','touching','hugging','kissing',
                     'jumping','climbing','mounted','chained'}
    if any(w in r for w in _ACTION_WORDS):
        return 'ACTION'
    return 'ATTRIBUTE'

def _clean_label(label):
    label = label.strip().lower()
    words = label.split()
    while words and words[0] in ('a','an','the'):
        words = words[1:]
    seen, clean = [], []
    for w in words:
        if w not in seen:
            seen.append(w); clean.append(w)
    return ' '.join(clean) if clean else label

def _core_noun(phrase):
    stop = {'a','an','the','of','in','on','at','is','its','some','with',
            'left','right','old','young','small','large','big','little'}
    words = [w for w in phrase.lower().split() if w not in stop and len(w) >= 2]
    return words[-1] if words else phrase.lower().strip()

def detect_objects(image, queries):
    if gdino_model is None:
        return []
    import torch
    text = '. '.join(queries) + '.'
    inputs = gdino_processor(images=image, text=text, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = gdino_model(**inputs)
    results = gdino_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold=DETECTION_THRESHOLD, text_threshold=DETECTION_THRESHOLD,
        target_sizes=[image.size[::-1]],
    )[0]
    W, H = image.size
    lkey = 'text_labels' if 'text_labels' in results else 'labels'
    dets = []
    for score, label, box in zip(results['scores'], results[lkey], results['boxes']):
        x1, y1, x2, y2 = box.tolist()
        dets.append((_clean_label(label), score.item(), [x1/W, y1/H, x2/W, y2/H]))
    return dets

def _find_best_bbox(entity, detections):
    """Return highest-score bbox [x1,y1,x2,y2] for entity among detections list."""
    core = _core_noun(entity)
    best_score, best_box = -1, None
    for label, score, box in detections:
        label_core = _core_noun(label)
        if core == label_core or core in label_core or label_core in core:
            if score > best_score:
                best_score, best_box = score, box
    return best_box

def _crop_to_bboxes(pil_image, box1, box2, padding=0.15):
    W, H = pil_image.size
    xs = [box1[0], box1[2], box2[0], box2[2]]
    ys = [box1[1], box1[3], box2[1], box2[3]]
    x1 = max(0.0, min(xs) - padding); y1 = max(0.0, min(ys) - padding)
    x2 = min(1.0, max(xs) + padding); y2 = min(1.0, max(ys) + padding)
    left, top, right, bottom = int(x1*W), int(y1*H), int(x2*W), int(y2*H)
    if right - left < 32 or bottom - top < 32:
        return pil_image
    return pil_image.crop((left, top, right, bottom))

def _spatial_verdict_from_boxes(subj_box, obj_box, rel):
    """True=geometry supports claim, False=contradicts, None=ambiguous.
    DEAD_ZONE: if positions are within 0.08 of each other, return None
    (too close to call deterministically — fall through to VQA).
    Only return False when geometry CLEARLY contradicts (opposite side).
    """
    cx_s = (subj_box[0] + subj_box[2]) / 2
    cy_s = (subj_box[1] + subj_box[3]) / 2
    cx_o = (obj_box[0] + obj_box[2]) / 2
    cy_o = (obj_box[1] + obj_box[3]) / 2
    r = rel.lower().strip()
    THRESH = 0.08   # min separation to make a deterministic call

    if r in ('left of','to the left of','to the left'):
        if cx_s < cx_o - THRESH: return True    # clearly to the left ✓
        if cx_s > cx_o + THRESH: return False   # clearly to the right ✗
        return None   # ambiguous — too close

    if r in ('right of','to the right of','to the right'):
        if cx_s > cx_o + THRESH: return True
        if cx_s < cx_o - THRESH: return False
        return None

    if r in ('above','over'):
        if cy_s < cy_o - THRESH: return True    # higher in image (lower y) ✓
        if cy_s > cy_o + THRESH: return False   # lower in image ✗
        return None

    if r in ('below','under','beneath','underneath'):
        if cy_s > cy_o + THRESH: return True
        if cy_s < cy_o - THRESH: return False
        return None

    if r in ('on top of','on'):
        # subject centroid above object AND horizontal overlap
        horiz_overlap = subj_box[0] < obj_box[2] and subj_box[2] > obj_box[0]
        if cy_s < cy_o - THRESH and horiz_overlap: return True
        if cy_s > cy_o + THRESH: return False   # subject is clearly below
        return None

    if r in ('in front of',):
        if cy_s > cy_o + THRESH: return True    # closer to camera ✓
        if cy_s < cy_o - THRESH: return False
        return None

    if r in ('behind','in back of'):
        if cy_s < cy_o - THRESH: return True
        if cy_s > cy_o + THRESH: return False
        return None

    return None   # relation not mappable to geometry (inside/outside etc.)

_COUNTERFACTUAL_MAP = {
    'riding':'standing next to','sitting on':'standing near',
    'holding':'standing next to','carrying':'walking away from',
    'wearing':'next to','eating':'looking at','pulling':'pushing',
    'pushing':'pulling','throwing':'holding','catching':'dropping',
    'driving':'standing near','leading':'following',
    'playing with':'ignoring','using':'near',
    'standing on':'next to','lying on':'sitting near',
    'hanging from':'standing near','leaning on':'standing near',
}

def _verify_triple(subj, rel, obj_, detections, pil_image):
    """
    Full RelCheck verifier: type-aware routing.
    SPATIAL  → GDino bbox geometry (deterministic)
    ACTION   → crop VQA (2 YES/NO + 1 contrastive forced-choice)
    ATTRIBUTE→ full-image VQA
    Returns True (correct), False (hallucination), None (uncertain).
    """
    rel_type = _classify_rel(rel)
    b64_full = None

    def _get_b64(pil):
        buf = _BytesIO()
        pil.convert('RGB').save(buf, format='JPEG', quality=85)
        return _b64.b64encode(buf.getvalue()).decode()

    # ── SPATIAL: geometry first ───────────────────────────────────────
    if rel_type == 'SPATIAL' and USE_GDINO and detections:
        subj_box = _find_best_bbox(subj, detections)
        obj_box  = _find_best_bbox(obj_, detections)
        if subj_box and obj_box:
            verdict = _spatial_verdict_from_boxes(subj_box, obj_box, rel)
            if verdict is not None:
                return verdict   # deterministic answer — trust it
        # Geometry ambiguous (inside/outside or missing boxes) → fall through to VQA

    # ── ACTION / ATTRIBUTE or spatial fallback: crop VQA ─────────────
    if pil_image is None:
        return None

    subj_box = _find_best_bbox(subj, detections) if detections else None
    obj_box  = _find_best_bbox(obj_, detections) if detections else None

    using_crop = bool(subj_box and obj_box)
    if using_crop:
        crop = _crop_to_bboxes(pil_image, subj_box, obj_box, padding=0.15)
        region_hint = 'this cropped region'
    else:
        crop = pil_image
        region_hint = 'the full image'
    crop_b64 = _get_b64(crop)

    # 2 standard YES/NO questions
    yes_v = no_v = 0
    for q in [
        f'In this image, is the {subj} {rel} the {obj_}? Look carefully at {region_hint}. Answer YES or NO only.',
        f'Does the {subj} appear to be {rel} the {obj_} here? Observe {region_hint} closely. Answer YES or NO only.',
    ]:
        r = vlm_call([{'role':'user','content':[
            {'type':'image_url','image_url':{'url':f'data:image/jpeg;base64,{crop_b64}'}},
            {'type':'text','text':q},
        ]}], max_tokens=5)
        if not r: continue
        rl = r.strip().lower()
        if 'yes' in rl and 'no' not in rl: yes_v += 1
        elif 'no' in rl: no_v += 1

    # 1 contrastive forced-choice (A/B randomised to remove position bias)
    cf_rel = _COUNTERFACTUAL_MAP.get(rel.lower(), f'not {rel}')
    ab_flip = (hash(f'{subj}{rel}{obj_}') % 2 == 1)
    opt_a, opt_b = (cf_rel, rel) if ab_flip else (rel, cf_rel)
    cf_q = (f'Look at {region_hint} carefully. Which is more accurate?\n'
            f'(A) The {subj} is {opt_a} the {obj_}\n'
            f'(B) The {subj} is {opt_b} the {obj_}\n'
            f'Answer ONLY the letter A or B.')
    cf_r = vlm_call([{'role':'user','content':[
        {'type':'image_url','image_url':{'url':f'data:image/jpeg;base64,{crop_b64}'}},
        {'type':'text','text':cf_q},
    ]}], max_tokens=5)
    if cf_r:
        cr = cf_r.strip().upper()
        chose_a = ('A' in cr and 'B' not in cr)
        chose_b = ('B' in cr and 'A' not in cr)
        if ab_flip:
            if chose_b: yes_v += 1   # chose original
            elif chose_a: no_v += 1  # chose counterfactual
        else:
            if chose_a: yes_v += 1
            elif chose_b: no_v += 1

    total = yes_v + no_v
    if total == 0: return None
    if yes_v > no_v: return True
    if no_v > yes_v: return False
    return None   # tie


# ── Load LLaVA locally (only when CAPTIONER == 'llava') ───────────────
llava_model_local = llava_processor_local = None
if CAPTIONER == 'llava':
    try:
        import torch
        from transformers import LlavaForConditionalGeneration, AutoProcessor as _AP
        _LLAVA_ID = 'llava-hf/llava-1.5-7b-hf'
        print(f'Loading LLaVA-1.5-7B locally on {DEVICE}...')
        llava_processor_local = _AP.from_pretrained(_LLAVA_ID)
        llava_model_local = LlavaForConditionalGeneration.from_pretrained(
            _LLAVA_ID, torch_dtype=torch.float16, device_map='auto'
        )
        print('  LLaVA-1.5-7B ready.')
    except Exception as e:
        print(f'  LLaVA local load failed: {e}')
        print('  Falling back to Together.ai (Llama-3.2-11B).')
        nb['cells'][1]  # no-op reference to avoid lint
        _CAPTIONER_MODELS['llava'] = 'meta-llama/Llama-3.2-11B-Vision-Instruct-Turbo'

def local_caption_llava(pil_image, max_new_tokens=300):
    '''Caption with local LLaVA-1.5-7B. Returns None if model not loaded.'''
    if llava_model_local is None:
        return None
    import torch
    conversation = [{'role': 'user', 'content': [
        {'type': 'image'},
        {'type': 'text', 'text': (
            'Describe this image in detail. Include all objects, '
            'their spatial positions relative to each other, any actions '
            'or interactions taking place, and notable attributes like colors and sizes.'
        )},
    ]}]
    prompt_text = llava_processor_local.apply_chat_template(
        conversation, add_generation_prompt=True
    )
    inputs = llava_processor_local(
        images=pil_image, text=prompt_text, return_tensors='pt'
    ).to(llava_model_local.device)
    with torch.no_grad():
        out = llava_model_local.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False
        )
    gen_ids = out[:, inputs['input_ids'].shape[1]:]
    return llava_processor_local.decode(gen_ids[0], skip_special_tokens=True).strip()

print('RelCheck verifier helpers loaded.')


In [ ]:
# ============================================================
# Cell 2 — Load VG relationships + select candidate images
# ============================================================
# Strategy:
#   1. Scan LOCAL_IMAGES_ZIP first → get available image IDs
#   2. Load all of relationships.json, keep only entries whose
#      image_id is in the available set  (no STREAM_LIMIT needed)
#   3. Score and rank those candidates

# VG data moved from Stanford to UW (Ranjay Krishna's server).
VG_REL_URLS = [
    'https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset/relationships_v1_2.json.zip',
    'https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset/relationships_v1.json.zip',
]
# Path to manually downloaded relationships zip on Drive (or None to auto-download)
LOCAL_ZIP_PATH = f'{SAVE_DIR}/relationships.json.zip'

SKIP_PREDS = {'has','have','of','is','are','was','were','with',
              'and','a','an','the','it','its','for','to','by',
              # Too vague — nearly always COMPATIBLE with any caption description
              'near','nearby','beside','close to','adjacent to',
              'around','between','among',
              'part of','made of','attached to','connected to',
              'belonging to','painted on','written on',
              'along','across','through','toward'}

# Relations specific enough to actually contradict a captioner.
# We score images higher for having more of these — they produce
# real, detectable hallucinations aligned with what RelCheck targets.
PREFERRED_PREDS = {
    # Directional spatial
    'left of','right of','above','below','under','on top of',
    'on','inside','outside','over','beneath','leaning on',
    # Action / interaction (clearly verifiable by VQA)
    'riding','holding','carrying','wearing','eating','drinking',
    'sitting on','standing on','lying on','walking on','jumping over',
    'kicking','touching','pulling','pushing','hugging','kissing',
    'feeding','reading','watching','playing with','using',
    'mounted on','chained to',
}

def clean_vg_name(name):
    name = name.lower().strip()
    for art in ['a ', 'an ', 'the ']:
        if name.startswith(art):
            name = name[len(art):]
    return name.strip()

# ── Step 1: scan the images zip to know which IDs we actually have ─────
zip_available_ids = set()
if LOCAL_IMAGES_ZIP and Path(LOCAL_IMAGES_ZIP).exists():
    print(f'Scanning {LOCAL_IMAGES_ZIP} for available image IDs...')
    with zipfile.ZipFile(LOCAL_IMAGES_ZIP, 'r') as zf:
        for name in zf.namelist():
            stem = Path(name).stem
            try:
                zip_available_ids.add(int(stem))
            except ValueError:
                pass
    print(f'  Found {len(zip_available_ids)} images in zip '
          f'(IDs {min(zip_available_ids)}–{max(zip_available_ids)})')
else:
    print('WARNING: LOCAL_IMAGES_ZIP not found. '
          'Set LOCAL_IMAGES_ZIP in Cell 1 to your images2.zip path.')

# ── Step 2: load relationships.json, filtered to available IDs ─────────

if Path(VG_DATA_PATH).exists():
    with open(VG_DATA_PATH) as f:
        vg_data = json.load(f)
    print(f'Loaded {len(vg_data)} VG entries from cache')
    # Warn if cache was built without zip filtering
    cached_ids = {e['image_id'] for e in vg_data}
    overlap = len(cached_ids & zip_available_ids) if zip_available_ids else len(cached_ids)
    if zip_available_ids and overlap == 0:
        print('  WARNING: cached data has no overlap with your zip — deleting cache.')
        Path(VG_DATA_PATH).unlink()
        vg_data = None
    else:
        print(f'  {overlap}/{len(cached_ids)} cached entries overlap with zip')
else:
    vg_data = None

if vg_data is None:
    buf = None
    if LOCAL_ZIP_PATH and Path(LOCAL_ZIP_PATH).exists():
        print(f'Found local relationships zip at {LOCAL_ZIP_PATH}')
        with open(LOCAL_ZIP_PATH, 'rb') as zf:
            buf = io.BytesIO(zf.read())
    else:
        print('Downloading VG relationships (trying multiple URLs)...')
        for url in VG_REL_URLS:
            try:
                print(f'  Trying {url}')
                resp = requests.get(url, stream=True, timeout=60)
                resp.raise_for_status()
                buf = io.BytesIO()
                total = 0
                for chunk in resp.iter_content(chunk_size=1024*1024):
                    buf.write(chunk)
                    total += len(chunk)
                    print(f'  {total/1e6:.1f} MB...', end='\r')
                print(f'  Downloaded {total/1e6:.1f} MB')
                break
            except Exception as e:
                print(f'  Failed: {e}')
                buf = None

    if buf is None:
        raise RuntimeError(
            'Could not load relationships.json.\n'
            'Please download from https://visualgenome.org/api/v0/api_home\n'
            f'and upload to Drive as: {LOCAL_ZIP_PATH}'
        )

    buf.seek(0)
    print('Parsing relationships.json...')
    with zipfile.ZipFile(buf) as zf:
        fname = next(n for n in zf.namelist() if n.endswith('.json'))
        raw = json.loads(zf.open(fname).read().decode('utf-8'))

    if zip_available_ids:
        vg_data = [e for e in raw if e['image_id'] in zip_available_ids]
        print(f'Filtered {len(raw)} total → {len(vg_data)} entries matching your zip')
    else:
        vg_data = raw[:5000]   # fallback if no zip scanned
        print(f'No zip scanned — using first {len(vg_data)} entries as fallback')

    with open(VG_DATA_PATH, 'w') as f:
        json.dump(vg_data, f)
    print(f'Saved {len(vg_data)} entries to cache')

# ── Step 3: score and rank candidates ─────────────────────────────────
candidates = []
for entry in vg_data:
    img_id = entry['image_id']
    rels   = entry.get('relationships', [])
    good   = [r for r in rels
              if r.get('predicate','').strip().lower() not in SKIP_PREDS
              and len(r.get('predicate','').split()) <= 4
              and r.get('subject',{}).get('name')
              and r.get('object',{}).get('name')]
    if len(good) < MIN_VG_RELS:
        continue
    # Score = preferred relation count (primary) + total distinct preds (tiebreak).
    # Preferred = specific spatial/action that captioners mention and can get wrong.
    # Vague relations like 'near', 'next to' are excluded (nearly always COMPATIBLE).
    pref_count = sum(1 for r in good
                     if r['predicate'].lower().strip() in PREFERRED_PREDS)
    total_preds = len({r['predicate'].lower() for r in good})
    score = pref_count * 100 + total_preds
    # Store only preferred + non-trivial relations as ground truth
    gt_rels = [r for r in good
               if r['predicate'].lower().strip() in PREFERRED_PREDS
               or r['predicate'].lower().strip() not in SKIP_PREDS]
    candidates.append((score, img_id, gt_rels[:10]))

candidates.sort(reverse=True)
candidates_by_id = {img_id: rels for _, img_id, rels in candidates}

print(f'\nTotal candidate images (>= {MIN_VG_RELS} good relations): {len(candidates_by_id)}')
if len(candidates_by_id) == 0:
    print('ERROR: 0 candidates found.')
    print(f'  zip_available_ids count: {len(zip_available_ids)}')
    print(f'  vg_data count: {len(vg_data)}')
    print('  Check that LOCAL_IMAGES_ZIP is the right zip (images2.zip = Part 2 = IDs 108250+).')
else:
    sample_ids = list(candidates_by_id.keys())[:5]
    print(f'  Sample IDs: {sample_ids}')
    for sc, iid, _ in candidates[:5]:
        pref = sc // 100
        print(f'    ID {iid}: {pref} preferred spatial/action rels (score={sc})')
    print(f'  Will select {min(N_IMAGES, len(candidates_by_id))} of these in Cell 3')

# ── Download image_data.json for Flickr URL fallback ──────────────────
IMAGE_DATA_PATH = f'{SAVE_DIR}/image_data.json'
IMAGE_DATA_URLS = [
    'https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset/image_data.json.zip',
]

if Path(IMAGE_DATA_PATH).exists():
    with open(IMAGE_DATA_PATH) as f:
        image_data = json.load(f)
    print(f'\nLoaded image_data.json from cache ({len(image_data)} entries)')
else:
    print('\nDownloading image_data.json...')
    img_buf = None
    for url in IMAGE_DATA_URLS:
        try:
            resp = requests.get(url, timeout=60)
            resp.raise_for_status()
            img_buf = io.BytesIO(resp.content)
            print(f'  Downloaded {len(resp.content)/1e6:.1f} MB')
            break
        except Exception as e:
            print(f'  Failed: {e}')
    if img_buf is None:
        print('  Could not download image_data.json — Flickr fallback disabled')
        image_data = []
    else:
        img_buf.seek(0)
        with zipfile.ZipFile(img_buf) as zf:
            fname = next(n for n in zf.namelist() if n.endswith('.json'))
            image_data = json.loads(zf.open(fname).read().decode('utf-8'))
        with open(IMAGE_DATA_PATH, 'w') as f:
            json.dump(image_data, f)
        print(f'  Saved {len(image_data)} entries')

flickr_urls = {}
for entry in image_data:
    img_id = entry.get('image_id') or entry.get('id')
    url = entry.get('flickr_300k_url') or entry.get('url', '')
    if img_id and url:
        flickr_urls[img_id] = url
print(f'Flickr URLs mapped: {len(flickr_urls)} images')


In [ ]:
# ============================================================
# Cell 3 — Download VG images + caption via Together.ai VLM
# ============================================================
# Captions are generated via Together.ai API (no local GPU needed).
# Uses a detailed describe prompt so we get rich relational captions
# — much more likely to surface detectable hallucinations than
# BLIP-2's terse 10-word outputs.

CAPTIONS_PATH = f'{SAVE_DIR}/{CAPTIONER}_captions.json'

DESCRIBE_PROMPT = (
    'Describe this image in detail. Include all objects, '
    'their spatial positions relative to each other, any actions '
    'or interactions taking place, and notable attributes like colors and sizes.'
)

def caption_image(pil_image, model=CAPTION_MODEL, max_tokens=300):
    '''Caption via local LLaVA if loaded, else Together.ai API.'''
    # Use local LLaVA when available (avoids Together.ai serverless restriction)
    if CAPTIONER == 'llava' and llava_model_local is not None:
        return local_caption_llava(pil_image, max_new_tokens=max_tokens)
    # Fallback: Together.ai API
    buf = BytesIO()
    pil_image.convert('RGB').save(buf, format='JPEG', quality=85)
    b64 = base64.b64encode(buf.getvalue()).decode()
    return llm_call(
        [{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{b64}'}},
            {'type': 'text', 'text': DESCRIBE_PROMPT},
        ]}],
        model=model, max_tokens=max_tokens,
    )

# ── Step 1: find available image IDs from zip or Flickr ─────────────
available_ids = set()
vg_zip = None   # will hold open ZipFile handle if using local zip

if LOCAL_IMAGES_ZIP and Path(LOCAL_IMAGES_ZIP).exists():
    print(f'Scanning zip: {LOCAL_IMAGES_ZIP} ...')
    vg_zip = zipfile.ZipFile(LOCAL_IMAGES_ZIP, 'r')
    for name in vg_zip.namelist():
        stem = Path(name).stem
        try:
            available_ids.add(int(stem))
        except ValueError:
            pass
    print(f'Found {len(available_ids)} images in zip')
else:
    available_ids = set(flickr_urls.keys())
    print(f'Using Flickr URLs: {len(available_ids)} images available')

# ── Step 2: select N_IMAGES from candidates that are available ───────
pool = [(score, img_id, rels) for score, img_id, rels in candidates
        if img_id in available_ids]
if len(pool) == 0:
    raise RuntimeError(
        'No candidates overlap with available images.\n'
        'Check that LOCAL_IMAGES_ZIP points to your VG_100K.zip or VG_100K_2.zip.'
    )
random.seed(42)
selected = random.sample(pool, min(N_IMAGES, len(pool)))
selected_images = {img_id: rels for _, img_id, rels in selected}
print(f'Selected {len(selected_images)}/{len(pool)} available candidates')

# ── Step 3: load selected images ─────────────────────────────────────
pil_images = {}
if vg_zip is not None:
    # Build filename→zippath lookup (handles subfolders inside zip)
    zip_index = {Path(n).stem: n for n in vg_zip.namelist() if n.endswith('.jpg')}
    for img_id in selected_images:
        zpath = zip_index.get(str(img_id))
        if zpath:
            try:
                pil_images[img_id] = Image.open(BytesIO(vg_zip.read(zpath))).convert('RGB')
            except Exception as e:
                print(f'  [{img_id}] load error: {e}')
        else:
            print(f'  [{img_id}] not found in zip')
else:
    print('Downloading selected images via Flickr URLs...')
    for img_id in selected_images:
        try:
            r = requests.get(flickr_urls[img_id], timeout=15)
            r.raise_for_status()
            pil_images[img_id] = Image.open(BytesIO(r.content)).convert('RGB')
        except Exception as e:
            print(f'  [{img_id}] failed: {e}')

print(f'Loaded {len(pil_images)}/{len(selected_images)} images')

# Generate captions — incremental: load existing, caption only missing images
if Path(CAPTIONS_PATH).exists():
    with open(CAPTIONS_PATH) as f:
        captions = {int(k): v for k, v in json.load(f).items()}
    print(f'Loaded {len(captions)} cached {CAPTIONER} captions')
else:
    captions = {}

missing = [img_id for img_id in pil_images if img_id not in captions]
if missing:
    print(f'Generating {len(missing)} new captions with {CAPTIONER} ({CAPTION_MODEL})...')
    for img_id in missing:
        pil = pil_images[img_id]
        cap = caption_image(pil)
        if cap:
            captions[img_id] = cap
            words = len(cap.split())
            print(f'  [{img_id}] {words}w: {cap[:80]}...')
        else:
            print(f'  [{img_id}] captioning failed')
    with open(CAPTIONS_PATH, 'w') as f:
        json.dump({str(k): v for k, v in captions.items()}, f)
    print(f'Total cached: {len(captions)} captions')
else:
    print(f'All {len(captions)} images already captioned.')

avg_words = sum(len(c.split()) for c in captions.values()) / max(len(captions), 1)
print(f'Avg caption length: {avg_words:.0f} words ({CAPTIONER})')
if captions:
    print('Example caption:')
    ex_id = list(captions.keys())[0]
    print(f'  [{ex_id}] {captions[ex_id][:200]}')
else:
    print('WARNING: No captions generated — check TOGETHER_API_KEY and CAPTIONER model ID')


In [ ]:
# ============================================================
# Cell 4 — Extract caption triples + identify GT hallucinations
# ============================================================
# A caption triple is a GT hallucination if:
#   - Same (subject, object) pair exists in VG (entity match)
#   - But the relation in the caption CONTRADICTS the VG relation
# We use Llama to judge compatibility — handles paraphrases cleanly.

TRIPLE_PROMPT = '''Extract relational claims from this caption as JSON.
Caption: "{caption}"
Output a JSON array: [{{"subject": "...", "relation": "...", "object": "..."}}]
Only claims about how two entities relate. ONLY valid JSON, no explanation.'''

# ── Co-occurrence trap table ─────────────────────────────────────────
# Key   = stereotypical caption relation (what captioners tend to hallucinate)
# Value = set of VG relations that CONTRADICT the caption claim
# If caption says key but VG says any value → GT hallucination
TRAP_TABLE = {
    # ── Action traps: captioner defaults to interaction but reality is just proximity ──
    'riding':       {'standing next to','next to','near','beside','behind','in front of',
                     'walking next to','standing by','walking by','standing beside',
                     'walking past','looking at','standing near','sitting on','sitting near','facing'},
    'sitting on':   {'standing next to','standing near','next to','near','beside',
                     'behind','in front of','standing behind','walking near','standing by','riding','facing'},
    'holding':      {'standing next to','next to','near','beside','in front of',
                     'standing near','looking at','standing by','walking near','facing'},
    'wearing':      {'next to','near','beside','holding','standing next to','carrying','facing'},
    'eating':       {'looking at','next to','near','beside','standing near','holding','facing'},
    'carrying':     {'standing next to','next to','near','beside','walking near',
                     'standing near','looking at','facing'},
    'playing with': {'next to','near','beside','looking at','standing near','standing next to','ignoring','avoiding','facing'},
    'feeding':      {'next to','near','beside','standing next to','ignoring','looking at','facing'},
    'petting':      {'next to','near','beside','standing next to','looking at','facing'},
    'pulling':      {'next to','near','beside','standing next to','pushing','facing'},
    'pushing':      {'next to','near','beside','standing next to','pulling','facing'},
    'touching':     {'standing next to','next to','near','beside','standing near','looking at'},
    'leaning on':   {'standing next to','next to','near','beside','standing near','behind'},
    'walking':      {'standing next to','standing near','sitting on','sitting near','lying on','facing'},
    'running':      {'standing next to','standing near','sitting on','sitting near','lying on','facing'},
    'lying on':     {'standing on','sitting on','standing next to','next to','near','standing near'},
    'hanging from': {'on','on top of','next to','near','standing on','sitting on'},
    'looking at':   {'behind','next to','near','beside','standing near','standing next to'},
    'watching':     {'behind','next to','near','beside','standing near','standing next to'},
    'facing':       {'behind','back to back','turned away from'},
    'standing on':  {'standing next to','next to','near','beside','standing near','walking near'},

    # ── Spatial traps: direct opposites ──
    'on':           {'under','below','beneath','beside','next to','near','outside'},
    'on top of':    {'under','below','beneath','beside','next to','near'},
    'above':        {'below','under','beneath','underneath','on ground'},
    'below':        {'above','on top of','over','on'},
    'under':        {'on','on top of','above','over','standing on'},
    'left of':      {'right of','to the right of'},
    'right of':     {'left of','to the left of'},
    'to the left of':  {'right of','to the right of'},
    'to the right of': {'left of','to the left of'},
    'in front of':  {'behind','in back of','back to back'},
    'behind':       {'in front of','facing'},
    'inside':       {'outside','next to','near','beside'},
    'outside':      {'inside','in','within','enclosed in'},
    'between':      {'left of','right of','behind','in front of'},
    'around':       {'inside','within','separated from'},
    'surrounding':  {'inside','within','separated from'},
}

# Reverse traps: VG has the stereotypical claim, caption has the non-stereotypical one
# e.g., VG says 'riding', caption says 'standing next to' — also a hallucination
_REVERSE_TRAPS = {}
for stereo, contras in TRAP_TABLE.items():
    for c in contras:
        _REVERSE_TRAPS.setdefault(c, set()).add(stereo)

# Synonym map: caption words → canonical VG labels
_ENT_SYNONYMS = {
    'man':'person','woman':'person','boy':'person','girl':'person',
    'child':'person','kid':'person','guy':'person','lady':'person',
    'men':'person','women':'person','people':'person','person':'person',
    'player':'person','rider':'person','driver':'person','worker':'person',
    'skater':'person','skier':'person','biker':'person','cyclist':'person',
    'dog':'dog','puppy':'dog','pup':'dog','hound':'dog',
    'cat':'cat','kitten':'cat','kitty':'cat',
    'bike':'bicycle','bicycle':'bicycle','cycle':'bicycle',
    'motorbike':'motorcycle','motorcycle':'motorcycle',
    'car':'car','vehicle':'car','automobile':'car','auto':'car',
    'truck':'truck','lorry':'truck','van':'truck',
    'horse':'horse','pony':'horse',
    'plane':'airplane','airplane':'airplane','aircraft':'airplane','jet':'airplane',
    'couch':'couch','sofa':'couch','settee':'couch',
    'tv':'television','television':'television','monitor':'television',
    'phone':'phone','cellphone':'phone','smartphone':'phone','mobile':'phone',
}

def _norm_entity(name):
    return ' '.join(_ENT_SYNONYMS.get(w, w) for w in name.lower().split())

def entity_overlap(a, b):
    a, b = _norm_entity(a), _norm_entity(b)
    STOP = {'a','an','the','of','in','on','at','is','its','some','with'}
    wa = {w for w in a.split() if w not in STOP and len(w) >= 3}
    wb = {w for w in b.split() if w not in STOP and len(w) >= 3}
    if not wa or not wb:
        return a.strip() == b.strip()
    for w in wa:
        for v in wb:
            if w == v:
                return True
            if len(w) >= 4 and len(v) >= 4:
                if w[:4] == v[:4]:        # prefix match (plurals, conjugations)
                    return True
                if w in v or v in w:      # substring containment
                    return True
    return False

GT_PATH     = f'{SAVE_DIR}/gt_hallucinations_{CAPTIONER}.json'
gt_hallucs  = {}   # img_id -> [{subject, cap_rel, object, vg_rel, claim}]
cap_triples = {}   # img_id -> [extracted triples]

# Set True to force re-run (e.g. after updating COMPAT_PROMPT)
FORCE_RERUN_GT = False
if not FORCE_RERUN_GT and Path(GT_PATH).exists():
    with open(GT_PATH) as f:
        saved = json.load(f)
    gt_hallucs  = {int(k): v for k, v in saved['hallucinations'].items()}
    cap_triples = {int(k): v for k, v in saved['cap_triples'].items()}
    print(f'Loaded from cache: {sum(len(v) for v in gt_hallucs.values())} GT hallucinations')
else:
    print('Extracting triples and matching against VG ground truth...')
    # ── Funnel counters ──────────────────────────────
    _f_triples   = 0   # total triples extracted
    _f_no_match  = 0   # triples with no VG entity match
    _f_compat    = 0   # triples matched + COMPATIBLE
    _f_contradict= 0   # triples matched + CONTRADICTORY
    # ────────────────────────────────────────────────
    for img_id, caption in captions.items():
        vg_rels = selected_images.get(img_id, [])
        raw = llm_call([{'role': 'user', 'content': TRIPLE_PROMPT.format(caption=caption)}],
                       max_tokens=400)
        try:
            clean = re.sub(r'```[a-z]*\n?', '', raw or '').replace('```', '').strip()
            triples = json.loads(clean)
        except Exception:
            triples = []
        cap_triples[img_id] = triples
        _f_triples += len(triples)

        hallucinations = []
        for ct in triples:
            cs = clean_vg_name(ct.get('subject', ''))
            cr = ct.get('relation', '').lower().strip()
            co = clean_vg_name(ct.get('object', ''))
            if not cs or not cr or not co:
                continue

            # Collect ALL matching VG relations for this entity pair
            # Check both directions: (caption_subj→VG_subj, caption_obj→VG_obj)
            # and also reversed (caption_subj→VG_obj, caption_obj→VG_subj)
            matching_vg = []
            for vr in vg_rels:
                vs = clean_vg_name(vr['subject']['name'])
                vp = vr['predicate'].lower().strip()
                vo = clean_vg_name(vr['object']['name'])
                if entity_overlap(cs, vs) and entity_overlap(co, vo):
                    matching_vg.append(vp)
                elif entity_overlap(cs, vo) and entity_overlap(co, vs):
                    matching_vg.append(vp)   # reversed entity order — still relevant

            if not matching_vg:
                _f_no_match += 1
                if _f_no_match <= 5:
                    print(f'    [no-match] ({cs}, {cr}, {co})')
                continue

            # 1. Exact or substring match → not a hallucination
            cr_norm = cr.lower().strip()
            exact = False
            for vp in matching_vg:
                if cr_norm == vp or cr_norm in vp or vp in cr_norm:
                    exact = True
                    break
            if exact:
                _f_compat += 1
                continue

            # 2. Check co-occurrence traps (forward: caption=stereotype, VG=reality)
            trap_contras = TRAP_TABLE.get(cr_norm, set())
            contradicting = [vp for vp in matching_vg if vp in trap_contras]

            # 3. Also check reverse traps (caption=non-stereotype, VG=stereotype)
            if not contradicting:
                rev_contras = _REVERSE_TRAPS.get(cr_norm, set())
                contradicting = [vp for vp in matching_vg if vp in rev_contras]

            if contradicting:
                _f_contradict += 1
                vg_rel = contradicting[0]
                hallucinations.append({
                    'subject': cs, 'cap_rel': cr, 'object': co,
                    'vg_rel': vg_rel, 'claim': f'{cs} {cr} {co}',
                })
                print(f'  [{img_id}] HALLUC: ({cs}, {cr}, {co}) — VG: "{vg_rel}"')
            else:
                _f_compat += 1   # matched but no trap triggered → assume fine
                if _f_compat <= 5:
                    print(f'    [no-trap] ({cs}, {cr}, {co}) VG: {matching_vg}')
        gt_hallucs[img_id] = hallucinations

    # ── Funnel stats ──────────────────────────────────────────
    _f_kept = sum(len(v) for v in gt_hallucs.values())
    print(f'\n── Funnel stats ──────────────────────────────')
    print(f'  Caption triples extracted   : {_f_triples}')
    print(f'  No VG entity match (dropped): {_f_no_match}')
    print(f'  Had VG match → COMPATIBLE   : {_f_compat}')
    print(f'  Had VG match → CONTRADICTORY: {_f_contradict}')
    print(f'  GT hallucinations kept      : {_f_kept}')
    print(f'──────────────────────────────────────────────')
    with open(GT_PATH, 'w') as f:
        json.dump({'hallucinations': {str(k): v for k, v in gt_hallucs.items()}, 'captioner': CAPTIONER,
                   'cap_triples':    {str(k): v for k, v in cap_triples.items()}}, f)


HTML_PATH = f'{SAVE_DIR}/triple_matching_report_{CAPTIONER}.html'

html_content = f'''<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>RelCheck VG Report</title>
<style>
body {{ font-family: Arial; margin: 20px; background: #f5f5f5; }}
.block {{ background: white; margin: 20px 0; padding: 15px; border-radius: 8px; border-left: 4px solid #0066cc; }}
.img_id {{ font-weight: bold; color: #0066cc; font-size: 16px; }}
.caption {{ background: #fffacd; padding: 10px; margin: 10px 0; border-radius: 4px; }}
.triples {{ display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin: 10px 0; }}
.col h4 {{ margin: 5px 0; font-size: 12px; text-transform: uppercase; color: #666; }}
.triple {{ background: #f0f0f0; padding: 6px 10px; margin: 3px 0; border-radius: 3px; font-family: monospace; font-size: 12px; }}
.matched {{ background: #d4edda; border-left: 3px solid #28a745; }}
.unmatched {{ background: #f8d7da; border-left: 3px solid #dc3545; }}
.vg-rels {{ background: #e7f3ff; padding: 10px; margin: 10px 0; border-left: 3px solid #0066cc; border-radius: 4px; }}
.hallucination {{ background: #fff3cd; padding: 8px; margin: 5px 0; border-left: 3px solid #ff9800; }}
</style></head><body>
<h1>RelCheck VG Matching Report</h1>
<p>Captioner: <b>{CAPTIONER}</b></p>
'''

for img_id in sorted(captions.keys()):
    cap = captions[img_id]
    triples = cap_triples.get(img_id, [])
    vg_rels = selected_images.get(img_id, [])
    hallucs = gt_hallucs.get(img_id, [])
    
    # Separate matched/unmatched
    matched_triples = []
    unmatched_triples = []
    for ct in triples:
        cs = clean_vg_name(ct.get('subject', ''))
        cr = ct.get('relation', '').lower().strip()
        co = clean_vg_name(ct.get('object', ''))
        if not cs or not cr or not co:
            continue
        found_match = False
        for vr in vg_rels:
            vs = clean_vg_name(vr['subject']['name'])
            vp = vr['predicate'].lower().strip()
            vo = clean_vg_name(vr['object']['name'])
            if entity_overlap(cs, vs) and entity_overlap(co, vo):
                found_match = True
                break
        triple_str = f'({cs}, {cr}, {co})'
        if found_match:
            matched_triples.append(triple_str)
        else:
            unmatched_triples.append(triple_str)
    
    # Build HTML for this image
    html_content += f'<div class="block"><div class="img_id">Image {img_id}</div>'
    html_content += f'<div class="caption">{cap}</div>'
    
    html_content += '<div class="triples"><div class="col"><h4>✓ Matched ({len(matched_triples)})</h4>'
    for t in matched_triples:
        html_content += f'<div class="triple matched">{t}</div>'
    html_content += '</div><div class="col"><h4>✗ Unmatched ({len(unmatched_triples)})</h4>'
    for t in unmatched_triples:
        html_content += f'<div class="triple unmatched">{t}</div>'
    html_content += '</div></div>'
    
    if vg_rels:
        html_content += '<div class="vg-rels"><h4>VG Relations:</h4>'
        for vr in vg_rels[:15]:
            vs = clean_vg_name(vr['subject']['name'])
            vp = vr['predicate'].lower().strip()
            vo = clean_vg_name(vr['object']['name'])
            html_content += f'<code>({vs}, {vp}, {vo})</code> '
        if len(vg_rels) > 15:
            html_content += f'<br><small>... and {len(vg_rels)-15} more</small>'
        html_content += '</div>'
    
    if hallucs:
        html_content += '<h4>🚨 Hallucinations:</h4>'
        for h in hallucs:
            html_content += f'<div class="hallucination">({h["subject"]}, {h["cap_rel"]}, {h["object"]}) → VG: "{h["vg_rel"]}"</div>'
    html_content += '</div>'

html_content += '</body></html>'

with open(HTML_PATH, 'w', encoding='utf-8') as f:
    f.write(html_content)
print(f'Saved HTML report: {HTML_PATH}')

total_hall = sum(len(v) for v in gt_hallucs.values())
n_affected = sum(1 for v in gt_hallucs.values() if v)
print(f'\nGT hallucinations: {total_hall} across {n_affected}/{len(captions)} images')
if total_hall == 0:
    print('WARNING: 0 hallucinations found. Try N_IMAGES=50 or lowering MIN_VG_RELS.')


In [ ]:
# ============================================================
# Cell 5 — RelCheck detection + recall  (full pipeline)
# ============================================================
# Per GT hallucination: run the actual RelCheck verifier.
#   SPATIAL  → GroundingDINO bbox geometry (deterministic)
#   ACTION   → crop VQA + contrastive forced-choice (Maverick-style)
#   fallback → full-image VQA
# USE_GDINO=False → plain 3-question VQA for everything (no GPU needed).

RECALL_PATH = f'{SAVE_DIR}/recall_results_{CAPTIONER}.json'

recall_results = []
total_hall = sum(len(v) for v in gt_hallucs.values())
print(f'Running RelCheck detection on {total_hall} GT hallucinations...\n')

for img_id, hallucs in gt_hallucs.items():
    if not hallucs:
        continue
    pil     = pil_images.get(img_id)
    caption = captions.get(img_id, '')

    # Run GDino detection on this image (get bboxes for all entities)
    if USE_GDINO and pil is not None and gdino_model is not None:
        all_entities = []
        for h in hallucs:
            all_entities += [h['subject'], h['object']]
        detections = detect_objects(pil, list(set(all_entities)))
    else:
        detections = []

    for hall in hallucs:
        s, r, o = hall['subject'], hall['cap_rel'], hall['object']
        rel_type = _classify_rel(r)
        verdict = _verify_triple(s, r, o, detections, pil)

        # verdict: True=caption is correct (MISSED), False=hallucination (DETECTED), None=uncertain
        detected = (verdict is False)

        result = {
            'img_id': img_id, 'caption': caption,
            'subject': s, 'cap_rel': r, 'object': o,
            'vg_rel': hall['vg_rel'],
            'rel_type': rel_type,
            'verdict': str(verdict),
            'detected': detected,
        }
        recall_results.append(result)
        status = 'DETECTED' if detected else ('MISSED(uncertain)' if verdict is None else 'MISSED')
        print(f'  [{img_id}] {status} [{rel_type}] ({s}, {r}, {o}) VG="{hall["vg_rel"]}" verdict={verdict}')

with open(RECALL_PATH, 'w') as f:
    json.dump(recall_results, f, indent=2)

# ── Text summary ──────────────────────────────────────────────────────
n_total     = len(recall_results)
n_detected  = sum(1 for r in recall_results if r['detected'])
n_uncertain = sum(1 for r in recall_results if r['verdict'] == 'None')
recall      = n_detected / max(n_total, 1)

print('\n' + '='*60)
print('RELCHECK RECALL — VISUAL GENOME GROUND TRUTH')
print('='*60)
print(f'  Verifier mode     : {"GDino + crop VQA + contrastive" if USE_GDINO else "plain VQA"}')
print(f'  GT hallucinations : {n_total}')
print(f'  Detected          : {n_detected}')
print(f'  Uncertain (None)  : {n_uncertain}')
print(f'  Recall            : {recall:.1%}')
print()
for rel_type in ['SPATIAL','ACTION','ATTRIBUTE']:
    sub = [r for r in recall_results if r['rel_type'] == rel_type]
    if sub:
        det = sum(1 for r in sub if r['detected'])
        print(f'  {rel_type:10s}: {det}/{len(sub)} detected')
print()
print('MISSED cases:')
missed = [r for r in recall_results if not r['detected']]
if not missed:
    print('  None — perfect recall!')
else:
    for r in missed:
        print(f'  [{r["img_id"]}] [{r["rel_type"]}] ({r["subject"]}, {r["cap_rel"]}, {r["object"]}) ')
        print(f'    VG="{r["vg_rel"]}"  verdict={r["verdict"]}')
print('='*60)

# ── Visual results ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

seen = set()
ordered_ids = [r['img_id'] for r in recall_results
               if not (r['img_id'] in seen or seen.add(r['img_id']))]

for img_id in ordered_ids:
    pil      = pil_images.get(img_id)
    caption  = captions.get(img_id, '(no caption)')
    img_results = [r for r in recall_results if r['img_id'] == img_id]
    if pil is None:
        continue

    fig, (ax_img, ax_txt) = plt.subplots(1, 2, figsize=(13, 4),
                                          gridspec_kw={'width_ratios': [1, 1]})
    ax_img.imshow(pil)
    ax_img.axis('off')
    ax_img.set_title(f'Image {img_id}', fontsize=9)
    ax_txt.axis('off')

    lines = []
    words = caption.split()
    for j in range(0, len(words), 10):
        lines.append(' '.join(words[j:j+10]))
    lines.append('')
    for r in img_results:
        icon = '✓ DETECTED' if r['detected'] else ('? UNCERTAIN' if r['verdict']=='None' else '✗ MISSED')
        lines.append(f'{icon}  [{r["rel_type"]}]')
        lines.append(f'  caption: {r["subject"]} {r["cap_rel"]} {r["object"]}')
        lines.append(f'  VG GT:   {r["subject"]} {r["vg_rel"]} {r["object"]}')
        lines.append(f'  verdict: {r["verdict"]}')
        lines.append('')

    ax_txt.text(0.02, 0.98, '\n'.join(lines),
                transform=ax_txt.transAxes,
                fontsize=8.5, verticalalignment='top', fontfamily='monospace')
    plt.tight_layout()
    plt.show()
    plt.close()
